# Лабораторная работа №5: Поиск экстремума функции одной переменной с помощью ГА с бинарным представлением особей

**Цель работы:** реализовать простой генетический алгоритм (ГА) с бинарным кодированием для нахождения максимума функции одной переменной.

**Задачи:**
1. Реализовать бинарное кодирование/декодирование вещественного числа из отрезка $[a, b]$.
2. Реализовать классические операторы ГА: пропорциональную селекцию (рулетка), одноточечный кроссовер, побитовую мутацию.
3. Провести поиск максимума функции $f(t) = (t - 1{,}3) \cdot \sin(0{,}5 \pi t + 1)$ на отрезке $[0, 7]$.
4. Визуализировать процесс работы ГА: график функции, динамику сходимости, анимацию популяции.
5. Исследовать влияние гиперпараметров (размер популяции, длина хромосомы, вероятности кроссовера и мутации, число поколений).

**Вариант 3.**

In [1]:
import numpy as np
import plotly.graph_objects as go
from plotly.subplots import make_subplots
from pathlib import Path

OUTPUTS = Path("report/outputs")
OUTPUTS.mkdir(exist_ok=True)

np.random.seed(42)

## 1. Целевая функция

$$f(t) = (t - 1{,}3) \cdot \sin(0{,}5 \pi t + 1), \quad t \in [0,\; 7]$$

Требуется найти **максимум** данной функции с помощью генетического алгоритма.

In [2]:
A, B = 0.0, 7.0


def target_function(t: np.ndarray | float) -> np.ndarray | float:
    return (t - 1.3) * np.sin(0.5 * np.pi * t + 1)

In [3]:
t_dense = np.linspace(A, B, 1000)
f_dense = target_function(t_dense)

fig = go.Figure()
fig.add_trace(go.Scatter(x=t_dense, y=f_dense, mode="lines", name="f(t)",
                         line=dict(color="royalblue", width=2.5)))

idx_max = np.argmax(f_dense)
fig.add_trace(go.Scatter(
    x=[t_dense[idx_max]], y=[f_dense[idx_max]],
    mode="markers+text", name="Аналитический максимум",
    marker=dict(color="red", size=12, symbol="star"),
    text=[f"t≈{t_dense[idx_max]:.3f}, f≈{f_dense[idx_max]:.3f}"],
    textposition="top center"
))

fig.update_layout(
    title="Целевая функция f(t) = (t − 1.3)·sin(0.5πt + 1)",
    xaxis_title="t", yaxis_title="f(t)",
    template="plotly_white", width=900, height=500
)
fig.write_html(str(OUTPUTS / "01_function_plot.html"))
fig.show()
print(f"Приблизительный максимум по сетке: t = {t_dense[idx_max]:.4f}, f(t) = {f_dense[idx_max]:.4f}")

Приблизительный максимум по сетке: t = 4.4915, f(t) = 3.1271


## 2. Бинарное кодирование / декодирование

Для представления вещественного числа $t \in [a, b]$ используется двоичная строка длины $L$.

**Декодирование:** $t = a + \frac{\text{decimal}(\text{chromosome})}{2^L - 1} \cdot (b - a)$

**Кодирование:** $\text{decimal} = \text{round}\left(\frac{t - a}{b - a} \cdot (2^L - 1)\right)$, затем перевод в двоичное представление.

In [4]:
def decode_chromosome(chromosome: np.ndarray, a: float, b: float) -> float:
    """Декодирует бинарную хромосому в вещественное число из [a, b]."""
    L = len(chromosome)
    decimal_val = 0
    for bit in chromosome:
        decimal_val = decimal_val * 2 + int(bit)
    return a + decimal_val * (b - a) / (2**L - 1)


def decode_population(population: np.ndarray, a: float, b: float) -> np.ndarray:
    """Декодирует массив хромосом в массив вещественных значений."""
    L = population.shape[1]
    powers = 2 ** np.arange(L - 1, -1, -1)
    decimal_vals = population @ powers
    return a + decimal_vals * (b - a) / (2**L - 1)


def encode_value(t: float, a: float, b: float, L: int) -> np.ndarray:
    """Кодирует вещественное число из [a, b] в бинарную хромосому длины L."""
    decimal_val = int(round((t - a) / (b - a) * (2**L - 1)))
    binary_str = format(decimal_val, f'0{L}b')
    return np.array([int(bit) for bit in binary_str], dtype=np.int8)

In [5]:
test_chrom = encode_value(3.5, A, B, 16)
test_decoded = decode_chromosome(test_chrom, A, B)
print(f"Кодирование t=3.5 → {test_chrom}")
print(f"Декодирование обратно → t={test_decoded:.6f}")
print(f"Погрешность: {abs(3.5 - test_decoded):.2e}")

Кодирование t=3.5 → [1 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0]
Декодирование обратно → t=3.500053
Погрешность: 5.34e-05


## 3. Операторы генетического алгоритма

### 3.1. Инициализация популяции
Случайная генерация $N$ бинарных строк длины $L$.

### 3.2. Пропорциональная селекция (рулетка)
Вероятность выбора особи пропорциональна её приспособленности. Для работы с отрицательными значениями фитнеса используем сдвиг: $\text{fitness}'_i = \text{fitness}_i - \min(\text{fitness}) + \varepsilon$.

### 3.3. Одноточечный кроссовер
Случайная точка разреза; потомки получают части от двух родителей.

### 3.4. Побитовая мутация
Каждый бит с заданной вероятностью $P_m$ инвертируется.

In [6]:
def init_population(pop_size: int, chrom_length: int) -> np.ndarray:
    """Инициализация случайной популяции."""
    return np.random.randint(0, 2, size=(pop_size, chrom_length), dtype=np.int8)


def roulette_selection(population: np.ndarray, fitness: np.ndarray) -> np.ndarray:
    """Пропорциональная селекция методом рулетки."""
    shifted = fitness - fitness.min() + 1e-8
    probs = shifted / shifted.sum()
    indices = np.random.choice(len(population), size=len(population), p=probs)
    return population[indices].copy()


def single_point_crossover(population: np.ndarray, p_cross: float) -> np.ndarray:
    """Одноточечный кроссовер."""
    pop = population.copy()
    n, L = pop.shape
    for i in range(0, n - 1, 2):
        if np.random.rand() < p_cross:
            point = np.random.randint(1, L)
            pop[i, point:], pop[i + 1, point:] = (
                pop[i + 1, point:].copy(),
                pop[i, point:].copy(),
            )
    return pop


def bitwise_mutation(population: np.ndarray, p_mut: float) -> np.ndarray:
    """Побитовая мутация."""
    pop = population.copy()
    mask = np.random.rand(*pop.shape) < p_mut
    pop[mask] = 1 - pop[mask]
    return pop

## 4. Основной цикл генетического алгоритма

In [7]:
def genetic_algorithm(
    func,
    a: float,
    b: float,
    pop_size: int = 60,
    chrom_length: int = 20,
    p_cross: float = 0.85,
    p_mut: float = 0.01,
    n_generations: int = 200,
    seed: int | None = 42,
    track_history: bool = False,
) -> dict:
    if seed is not None:
        np.random.seed(seed)

    population = init_population(pop_size, chrom_length)
    t_values = decode_population(population, a, b)
    fitness = func(t_values)

    best_fitness_history = []
    avg_fitness_history = []
    population_history = [] if track_history else None

    global_best_chrom = population[np.argmax(fitness)].copy()
    global_best_fit = fitness.max()

    for gen in range(n_generations):
        best_fitness_history.append(fitness.max())
        avg_fitness_history.append(fitness.mean())

        if track_history:
            population_history.append(t_values.copy())

        selected = roulette_selection(population, fitness)
        offspring = single_point_crossover(selected, p_cross)
        offspring = bitwise_mutation(offspring, p_mut)

        t_values_new = decode_population(offspring, a, b)
        fitness_new = func(t_values_new)

        # Элитизм: сохраняем лучшую особь
        worst_idx = np.argmin(fitness_new)
        if global_best_fit > fitness_new[worst_idx]:
            offspring[worst_idx] = global_best_chrom.copy()
            t_values_new[worst_idx] = decode_chromosome(global_best_chrom, a, b)
            fitness_new[worst_idx] = global_best_fit

        population = offspring
        t_values = t_values_new
        fitness = fitness_new

        cur_best_idx = np.argmax(fitness)
        if fitness[cur_best_idx] > global_best_fit:
            global_best_fit = fitness[cur_best_idx]
            global_best_chrom = population[cur_best_idx].copy()

    best_fitness_history.append(fitness.max())
    avg_fitness_history.append(fitness.mean())
    if track_history:
        population_history.append(t_values.copy())

    best_t = decode_chromosome(global_best_chrom, a, b)
    best_f = global_best_fit

    return {
        "best_t": best_t,
        "best_f": best_f,
        "best_chromosome": global_best_chrom,
        "best_fitness_history": np.array(best_fitness_history),
        "avg_fitness_history": np.array(avg_fitness_history),
        "population_history": population_history,
    }

## 5. Запуск ГА с параметрами по умолчанию

In [8]:
result = genetic_algorithm(
    target_function, A, B,
    pop_size=60, chrom_length=20, p_cross=0.85, p_mut=0.01,
    n_generations=200, seed=42, track_history=True,
)

print("=" * 50)
print("Результат работы генетического алгоритма:")
print(f"  Лучшее t  = {result['best_t']:.6f}")
print(f"  f(t)      = {result['best_f']:.6f}")
print(f"  Хромосома = {''.join(map(str, result['best_chromosome']))}")
print("=" * 50)

Результат работы генетического алгоритма:
  Лучшее t  = 4.488825
  f(t)      = 3.127117
  Хромосома = 10100100001010011010


## 6. Визуализация сходимости

In [9]:
gens = np.arange(len(result["best_fitness_history"]))

fig = go.Figure()
fig.add_trace(go.Scatter(
    x=gens, y=result["best_fitness_history"],
    mode="lines", name="Лучший фитнес",
    line=dict(color="crimson", width=2)
))
fig.add_trace(go.Scatter(
    x=gens, y=result["avg_fitness_history"],
    mode="lines", name="Средний фитнес",
    line=dict(color="steelblue", width=2, dash="dash")
))
fig.update_layout(
    title="Сходимость ГА: лучший и средний фитнес по поколениям",
    xaxis_title="Поколение", yaxis_title="Значение f(t)",
    template="plotly_white", width=900, height=500,
    legend=dict(x=0.65, y=0.15)
)
fig.write_html(str(OUTPUTS / "02_convergence.html"))
fig.show()

## 7. Анимация популяции на графике функции

In [10]:
history = result["population_history"]
n_frames = len(history)
step = max(1, n_frames // 40)
frame_indices = list(range(0, n_frames, step))
if frame_indices[-1] != n_frames - 1:
    frame_indices.append(n_frames - 1)

frames = []
for i in frame_indices:
    t_pop = history[i]
    f_pop = target_function(t_pop)
    frames.append(go.Frame(
        data=[
            go.Scatter(x=t_dense, y=f_dense, mode="lines",
                       line=dict(color="royalblue", width=2), name="f(t)"),
            go.Scatter(x=t_pop, y=f_pop, mode="markers",
                       marker=dict(color="red", size=8, opacity=0.7),
                       name="Особи")
        ],
        name=str(i),
        layout=go.Layout(title_text=f"Популяция — поколение {i}")
    ))

t_init = history[0]
f_init = target_function(t_init)

fig = go.Figure(
    data=[
        go.Scatter(x=t_dense, y=f_dense, mode="lines",
                   line=dict(color="royalblue", width=2), name="f(t)"),
        go.Scatter(x=t_init, y=f_init, mode="markers",
                   marker=dict(color="red", size=8, opacity=0.7),
                   name="Особи")
    ],
    frames=frames,
)

slider_steps = [
    dict(args=[[str(i)], dict(frame=dict(duration=200, redraw=True), mode="immediate")],
         label=str(i), method="animate")
    for i in frame_indices
]

fig.update_layout(
    title="Анимация популяции на графике функции (поколение 0)",
    xaxis_title="t", yaxis_title="f(t)",
    template="plotly_white", width=950, height=550,
    updatemenus=[dict(
        type="buttons", showactive=False,
        x=0.05, y=1.12, xanchor="left",
        buttons=[
            dict(label="▶ Запуск", method="animate",
                 args=[None, dict(frame=dict(duration=200, redraw=True),
                                  fromcurrent=True, mode="immediate")]),
            dict(label="⏸ Пауза", method="animate",
                 args=[[None], dict(frame=dict(duration=0, redraw=False),
                                    mode="immediate")])
        ]
    )],
    sliders=[dict(active=0, steps=slider_steps, currentvalue=dict(prefix="Поколение: "))]
)
fig.write_html(str(OUTPUTS / "03_population_animation.html"))
fig.show()

## 8. Исследование влияния гиперпараметров

Проведём серию экспериментов, варьируя по одному параметру при фиксированных остальных.

| Параметр | Значения |
|---|---|
| Размер популяции | 20, 40, 60, 100 |
| Длина хромосомы | 10, 16, 20, 24 |
| P кроссовера | 0.5, 0.7, 0.85, 0.95 |
| P мутации | 0.001, 0.01, 0.05, 0.1 |
| Число поколений | 50, 100, 200, 500 |

In [11]:
DEFAULT_PARAMS = dict(
    pop_size=60, chrom_length=20, p_cross=0.85, p_mut=0.01, n_generations=200
)

PARAM_GRID = {
    "pop_size": [20, 40, 60, 100],
    "chrom_length": [10, 16, 20, 24],
    "p_cross": [0.5, 0.7, 0.85, 0.95],
    "p_mut": [0.001, 0.01, 0.05, 0.1],
    "n_generations": [50, 100, 200, 500],
}

PARAM_LABELS = {
    "pop_size": "Размер популяции",
    "chrom_length": "Длина хромосомы",
    "p_cross": "P кроссовера",
    "p_mut": "P мутации",
    "n_generations": "Число поколений",
}

N_RUNS = 10

param_results = {}

for param_name, values in PARAM_GRID.items():
    param_results[param_name] = {}
    for val in values:
        run_params = DEFAULT_PARAMS.copy()
        run_params[param_name] = val
        best_fits = []
        best_ts = []
        convergence_curves = []
        for run_i in range(N_RUNS):
            res = genetic_algorithm(
                target_function, A, B, seed=run_i * 7 + 13, **run_params
            )
            best_fits.append(res["best_f"])
            best_ts.append(res["best_t"])
            convergence_curves.append(res["best_fitness_history"])
        param_results[param_name][val] = {
            "best_fits": np.array(best_fits),
            "best_ts": np.array(best_ts),
            "convergence": convergence_curves,
            "mean_fit": np.mean(best_fits),
            "std_fit": np.std(best_fits),
        }

print("Эксперименты завершены.")

Эксперименты завершены.


In [12]:
fig = make_subplots(
    rows=3, cols=2,
    subplot_titles=[
        PARAM_LABELS[p] for p in PARAM_GRID
    ] + [""],
    vertical_spacing=0.10, horizontal_spacing=0.08
)

colors = ["#636EFA", "#EF553B", "#00CC96", "#AB63FA", "#FFA15A"]

positions = [(1, 1), (1, 2), (2, 1), (2, 2), (3, 1)]

for idx, (param_name, values) in enumerate(PARAM_GRID.items()):
    row, col = positions[idx]
    means = [param_results[param_name][v]["mean_fit"] for v in values]
    stds = [param_results[param_name][v]["std_fit"] for v in values]
    labels = [str(v) for v in values]

    fig.add_trace(go.Bar(
        x=labels, y=means,
        error_y=dict(type="data", array=stds, visible=True),
        marker_color=colors[idx],
        name=PARAM_LABELS[param_name],
        showlegend=False,
    ), row=row, col=col)

    fig.update_xaxes(title_text=PARAM_LABELS[param_name], row=row, col=col)
    fig.update_yaxes(title_text="Средний макс. f(t)", row=row, col=col)

fig.update_layout(
    title_text="Влияние гиперпараметров на качество решения (среднее ± σ по 10 запускам)",
    template="plotly_white", width=1050, height=900,
    showlegend=False
)
fig.write_html(str(OUTPUTS / "04_param_analysis.html"))
fig.show()

### Кривые сходимости для различных параметров

In [13]:
fig = make_subplots(
    rows=3, cols=2,
    subplot_titles=[
        f"Сходимость: {PARAM_LABELS[p]}" for p in PARAM_GRID
    ] + [""],
    vertical_spacing=0.10, horizontal_spacing=0.08
)

line_styles = ["solid", "dash", "dot", "dashdot"]

for idx, (param_name, values) in enumerate(PARAM_GRID.items()):
    row, col = positions[idx]
    for j, val in enumerate(values):
        curves = param_results[param_name][val]["convergence"]
        max_len = max(len(c) for c in curves)
        avg_curve = np.zeros(max_len)
        counts = np.zeros(max_len)
        for c in curves:
            avg_curve[:len(c)] += c
            counts[:len(c)] += 1
        avg_curve /= counts

        fig.add_trace(go.Scatter(
            x=np.arange(max_len), y=avg_curve,
            mode="lines", name=f"{val}",
            line=dict(dash=line_styles[j % len(line_styles)], width=1.8),
            legendgroup=param_name,
            showlegend=(idx == 0),
        ), row=row, col=col)

    fig.update_xaxes(title_text="Поколение", row=row, col=col)
    fig.update_yaxes(title_text="Лучший f(t)", row=row, col=col)

fig.update_layout(
    title_text="Кривые сходимости при варьировании гиперпараметров",
    template="plotly_white", width=1050, height=900,
)
fig.write_html(str(OUTPUTS / "05_param_analysis.html"), include_plotlyjs=True)
fig.show()

## 9. Сводная таблица результатов

In [14]:
header_vals = [
    "Параметр", "Значение", "Среднее f(t)", "σ f(t)",
    "Лучшее f(t)", "Лучшее t", "Худшее f(t)"
]

table_data = {h: [] for h in header_vals}

for param_name, values in PARAM_GRID.items():
    for val in values:
        info = param_results[param_name][val]
        table_data["Параметр"].append(PARAM_LABELS[param_name])
        table_data["Значение"].append(str(val))
        table_data["Среднее f(t)"].append(f"{info['mean_fit']:.4f}")
        table_data["σ f(t)"].append(f"{info['std_fit']:.4f}")
        table_data["Лучшее f(t)"].append(f"{info['best_fits'].max():.4f}")
        best_run_idx = np.argmax(info['best_fits'])
        table_data["Лучшее t"].append(f"{info['best_ts'][best_run_idx]:.4f}")
        table_data["Худшее f(t)"].append(f"{info['best_fits'].min():.4f}")

fill_colors = []
for i in range(len(table_data["Параметр"])):
    fill_colors.append("#f0f4ff" if i % 2 == 0 else "white")

fig = go.Figure(data=[go.Table(
    header=dict(
        values=[f"<b>{h}</b>" for h in header_vals],
        fill_color="#3366cc", font=dict(color="white", size=13),
        align="center", height=35
    ),
    cells=dict(
        values=[table_data[h] for h in header_vals],
        fill_color=[fill_colors],
        align="center", height=28, font=dict(size=12)
    )
)])

fig.update_layout(
    title="Сводная таблица результатов параметрического исследования",
    width=1100, height=700,
    margin=dict(l=20, r=20, t=50, b=20)
)
fig.write_html(str(OUTPUTS / "06_results_table.html"))
fig.show()

## 10. Итоговый результат

In [15]:
print("=" * 60)
print("ИТОГ: Поиск максимума функции f(t) = (t-1.3)·sin(0.5πt+1)")
print("=" * 60)
print(f"  Область поиска:  t ∈ [{A}, {B}]")
print(f"  Найденный максимум:")
print(f"    t*     = {result['best_t']:.6f}")
print(f"    f(t*)  = {result['best_f']:.6f}")
print(f"  Хромосома: {''.join(map(str, result['best_chromosome']))}")
print()
print("  Параметры ГА:")
print(f"    Размер популяции = {DEFAULT_PARAMS['pop_size']}")
print(f"    Длина хромосомы  = {DEFAULT_PARAMS['chrom_length']}")
print(f"    P кроссовера     = {DEFAULT_PARAMS['p_cross']}")
print(f"    P мутации        = {DEFAULT_PARAMS['p_mut']}")
print(f"    Число поколений  = {DEFAULT_PARAMS['n_generations']}")
print()

t_ref = t_dense[np.argmax(f_dense)]
f_ref = f_dense.max()
print(f"  Справочный максимум (по сетке 1000 точек):")
print(f"    t_ref  = {t_ref:.6f}")
print(f"    f_ref  = {f_ref:.6f}")
print(f"  Отклонение: |t* - t_ref| = {abs(result['best_t'] - t_ref):.6f}")
print("=" * 60)

ИТОГ: Поиск максимума функции f(t) = (t-1.3)·sin(0.5πt+1)
  Область поиска:  t ∈ [0.0, 7.0]
  Найденный максимум:
    t*     = 4.488825
    f(t*)  = 3.127117
  Хромосома: 10100100001010011010

  Параметры ГА:
    Размер популяции = 60
    Длина хромосомы  = 20
    P кроссовера     = 0.85
    P мутации        = 0.01
    Число поколений  = 200

  Справочный максимум (по сетке 1000 точек):
    t_ref  = 4.491491
    f_ref  = 3.127088
  Отклонение: |t* - t_ref| = 0.002666


## Выводы

1. Реализован генетический алгоритм с бинарным кодированием для поиска максимума функции одной переменной.
2. Использованы классические операторы: пропорциональная селекция (рулетка), одноточечный кроссовер, побитовая мутация, элитизм.
3. ГА успешно находит глобальный максимум целевой функции с высокой точностью.
4. Параметрическое исследование показало:
   - Увеличение размера популяции повышает надёжность, но замедляет работу.
   - Длина хромосомы определяет точность представления: при малой длине (10 бит) точность ограничена.
   - Вероятность кроссовера 0.7–0.85 обеспечивает наилучший баланс между исследованием и эксплуатацией.
   - Слишком высокая вероятность мутации (0.1) разрушает хорошие решения; оптимум — 0.01.
   - 100–200 поколений достаточно для сходимости при данных параметрах.